# Least-Privilege Ephemeral Identity | Agent Safety & Resilience

In [1]:
# Ephemeral Identity for Agent Tasks
import time
import secrets
from dataclasses import dataclass
from typing import Dict, Set, Optional

In [2]:
@dataclass
class EphemeralToken:
    token_id: str
    agent_name: str
    permissions: Set[str]
    created_at: float
    ttl_seconds: float

    @property
    def is_expired(self) -> bool:
        return time.time() - self.created_at > self.ttl_seconds

class IdentityService:
    def __init__(self):
        self._tokens: Dict[str, EphemeralToken] = {}

    def issue_token(self, agent: str, permissions: Set[str], ttl: float = 300) -> str:
        token_id = secrets.token_hex(16)
        self._tokens[token_id] = EphemeralToken(
            token_id=token_id, agent_name=agent,
            permissions=permissions, created_at=time.time(), ttl_seconds=ttl,
        )
        return token_id

    def check_permission(self, token_id: str, permission: str) -> bool:
        token = self._tokens.get(token_id)
        if not token or token.is_expired:
            return False
        return permission in token.permissions

    def revoke(self, token_id: str):
        self._tokens.pop(token_id, None)

In [3]:
# Usage
identity = IdentityService()

# Issue a scoped token for a data analysis task
token = identity.issue_token("data_agent", {"read_db", "write_report"}, ttl=300)

# Agent checks permissions before acting
print(f"Can read DB: {identity.check_permission(token, 'read_db')}")
print(f"Can delete DB: {identity.check_permission(token, 'delete_db')}")  # False!
print(f"Can write report: {identity.check_permission(token, 'write_report')}")

# Revoke after task completion
identity.revoke(token)
print(f"After revoke - Can read DB: {identity.check_permission(token, 'read_db')}")

Can read DB: True
Can delete DB: False
Can write report: True
After revoke - Can read DB: False
